# 03 — Two-Agent System

**Requires:** `GROQ_API_KEY` in `.env`

---

## What's new here?

So far we've had **one** agent doing everything.  
Real-world systems split work across **multiple specialised agents**.

This notebook builds a **Researcher → Writer** pipeline:

```
  User Topic
       │
       ▼
  ┌──────────────┐
  │  RESEARCHER  │  Gathers facts, key points, statistics
  └──────┬───────┘
         │  hands off research notes
         ▼
  ┌──────────────┐
  │    WRITER    │  Turns notes into a polished article
  └──────┬───────┘
         │
         ▼
    Final Article
```

**Key idea — handoff:**  
Each agent reads the work the previous agent left in the state  
and uses it as context for its own output.

**Concepts covered:**
| Concept | What it means |
|---------|---------------|
| Specialised agents | Each agent has a focused prompt and role |
| State handoff | Agent B reads Agent A's output from shared state |
| Sequential pipeline | Nodes run in a fixed order |
| System prompts | Each agent's personality is defined by its system prompt |

## Step 1 — Setup

In [ ]:
# %pip install langgraph langchain langchain-groq python-dotenv --quiet

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")
assert os.getenv("GROQ_API_KEY"), "Add GROQ_API_KEY to your .env file"
print("API key loaded.")

## Step 2 — Define the Shared State

Both agents read and write to the **same state object**.  
This is how Agent B knows what Agent A found.

In [ ]:
from typing import TypedDict, Optional


class ArticleState(TypedDict):
    topic: str              # User's input
    research_notes: str     # Researcher's output → Writer reads this
    article: str            # Writer's output → final result
    word_count_target: int  # How long the article should be


print("Shared state defined.")
print("Fields:", list(ArticleState.__annotations__.keys()))

## Step 3 — Create the LLM

Both agents use the **same underlying LLM** but with **different system prompts** —  
the prompt is what gives each agent its unique personality and role.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatGroq(model="llama3-8b-8192", temperature=0.7)
print(f"LLM ready: {llm.model_name}")

## Step 4 — Define the Two Agents

Each agent is a **node function** with its own system prompt.

In [ ]:
RESEARCHER_SYSTEM = """You are a research specialist. Your job is to gather 
comprehensive information on a topic.

When given a topic, produce structured research notes with:
- 5-7 key facts
- 2-3 interesting statistics or numbers
- Historical context (1-2 sentences)
- Current trends (1-2 sentences)
- 3 important sub-topics worth exploring

Format your output clearly with headers. Be factual and concise."""


WRITER_SYSTEM = """You are a professional content writer. Your job is to transform 
research notes into an engaging, well-structured article.

When given research notes:
- Write an attention-grabbing introduction
- Organize content into 3-4 clear sections with headers
- Use the facts and statistics naturally in the text
- Write a strong conclusion with a key takeaway
- Keep the tone informative but accessible

Do NOT copy-paste from the notes — synthesise them into flowing prose."""


def researcher_agent(state: ArticleState) -> dict:
    """Research the topic and produce structured notes."""
    print("  [Researcher] Gathering information...")

    messages = [
        SystemMessage(content=RESEARCHER_SYSTEM),
        HumanMessage(content=f"Research this topic thoroughly: {state['topic']}"),
    ]

    response = llm.invoke(messages)
    notes = response.content

    print(f"  [Researcher] Done. Notes: {len(notes)} characters.")
    return {"research_notes": notes}


def writer_agent(state: ArticleState) -> dict:
    """Turn the researcher's notes into a polished article."""
    print("  [Writer] Writing article...")

    messages = [
        SystemMessage(content=WRITER_SYSTEM),
        HumanMessage(
            content=(
                f"Write a {state['word_count_target']}-word article based on these notes:\n\n"
                f"{state['research_notes']}"
            )
        ),
    ]

    response = llm.invoke(messages)
    article = response.content

    word_count = len(article.split())
    print(f"  [Writer] Done. Article: {word_count} words.")
    return {"article": article}


print("Two agents defined: researcher_agent, writer_agent")

## Step 5 — Build the Pipeline Graph

This is a **sequential pipeline**: every run follows the same fixed path.

```
START → researcher_agent → writer_agent → END
```

In [ ]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(ArticleState)

graph.add_node("researcher", researcher_agent)
graph.add_node("writer", writer_agent)

graph.add_edge(START, "researcher")
graph.add_edge("researcher", "writer")
graph.add_edge("writer", END)

pipeline = graph.compile()
print("Pipeline compiled.")

## Step 6 — Run the Pipeline

In [ ]:
initial_state: ArticleState = {
    "topic": "The rise of renewable energy and its impact on the global economy",
    "research_notes": "",
    "article": "",
    "word_count_target": 300,
}

print(f"Topic: {initial_state['topic']}")
print("=" * 60)

result = pipeline.invoke(initial_state)

print("\n" + "=" * 60)
print("RESEARCH NOTES (Researcher's output)")
print("=" * 60)
print(result["research_notes"])

In [ ]:
print("=" * 60)
print("FINAL ARTICLE (Writer's output)")
print("=" * 60)
print(result["article"])

## Step 7 — See the Handoff in Action

Let's verify that the writer **actually used** the researcher's notes.

In [ ]:
# Extract key facts from research notes
research_lines = [l.strip() for l in result["research_notes"].split("\n") if l.strip()]
article_text = result["article"].lower()

print("Checking if key research terms appear in the article...\n")

# Find numbers/stats from research that made it into the article
import re
numbers_in_research = re.findall(r'\d+\.?\d*%?', result["research_notes"])
numbers_in_article  = re.findall(r'\d+\.?\d*%?', result["article"])

shared = set(numbers_in_research) & set(numbers_in_article)
print(f"Numbers from research used in article: {sorted(shared)}")
print(f"\nResearch → {len(result['research_notes'].split())} words")
print(f"Article  → {len(result['article'].split())} words")

## Step 8 — Try Different Topics

In [ ]:
def create_article(topic: str, words: int = 250) -> str:
    """Run the full pipeline for any topic."""
    state: ArticleState = {
        "topic": topic,
        "research_notes": "",
        "article": "",
        "word_count_target": words,
    }
    result = pipeline.invoke(state)
    return result["article"]


# Change this topic to anything you like
article = create_article("Artificial intelligence in healthcare", words=250)
print(article)

## Step 9 — Visualise

In [ ]:
from IPython.display import Image, display

try:
    display(Image(pipeline.get_graph().draw_mermaid_png()))
except Exception:
    print(pipeline.get_graph().draw_mermaid())

## Summary

| What you built | What it taught |
|---------------|----------------|
| `ArticleState` | Both agents share one state — that's how they communicate |
| `researcher_agent` | Produces `research_notes` for the next agent to read |
| `writer_agent` | Reads `research_notes` from state, produces `article` |
| Sequential edges | Fixed pipeline: always researcher → writer |

**Why is this better than one big LLM call?**  
Each agent has a **focused prompt** → better output quality.  
You can swap, upgrade, or debug each agent independently.

---

**Next →** [04 — Supervisor Pattern](../04-supervisor-pattern/supervisor_pattern.ipynb)  
A supervisor agent dynamically routes work to the right specialist instead of a fixed order.